# 00 语音合成 TTS：ModelScope 小模型 + VoxCPM2 大模型体验

目标：先用一个 ModelScope 上可下载运行的小型 TTS 模型把语音合成链路跑清楚，再切到参数更大的 VoxCPM2 看现代语音模型在多语言、音色设计、参考音频克隆和部署成本上的差异。

本教程选择：

| 层级 | 模型 | 适合学习什么 |
| --- | --- | --- |
| 小模型 | `damo/speech_sambert-hifigan_tts_zh-cn_16k` | 文本输入、ModelScope pipeline、Sambert 声学模型、HiFiGAN vocoder、16kHz WAV |
| 大模型 | `OpenBMB/VoxCPM2` | 2B 参数、48kHz 输出、多语言、voice design、reference audio cloning、CFG 和 diffusion steps |

VoxCPM2 权重较大，本 notebook 默认只运行小模型。要运行 VoxCPM2，把后面的 `RUN_VOXCPM2 = False` 改成 `True`。

## 1. 安装依赖

在仓库根目录运行：

```bash
pip install -r requirements.txt
pip install -r speech/requirements-speech.txt
```

模型权重会通过 ModelScope 下载并缓存到本机。默认模型 ID 是：

```text
SMALL_TTS_MODEL_ID=damo/speech_sambert-hifigan_tts_zh-cn_16k
VOXCPM2_MODEL_ID=OpenBMB/VoxCPM2
```

如果你要替换为其他 ModelScope 模型，可以在运行前设置同名环境变量。VoxCPM2 官方文档说明 `pip install voxcpm` 即可使用 Python API；模型加载时会自动选择 `cuda -> mps -> cpu`，也可以显式指定设备。

In [ ]:
from pathlib import Path
import os
import shutil
import time

import soundfile as sf
import torch

OUTPUT_DIR = Path("speech/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_SMALL_TTS = True
RUN_VOXCPM2 = False

SMALL_TTS_MODEL_ID = os.getenv("SMALL_TTS_MODEL_ID", "damo/speech_sambert-hifigan_tts_zh-cn_16k")
VOXCPM2_MODEL_ID = os.getenv("VOXCPM2_MODEL_ID", "OpenBMB/VoxCPM2")
SMALL_TEXT = "语音合成会把文本转换成可以播放的波形。"
VOXCPM_TEXT = "(年轻女性，温柔自然，语速适中) 你好，这是 VoxCPM2 的语音合成示例。"

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("mps available:", getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
print("small model:", SMALL_TTS_MODEL_ID)
print("voxcpm2 model:", VOXCPM2_MODEL_ID)

## 2. 公共工具：设备选择、参数量、RTF

语音部署里除了“听起来好不好”，还要看两个基础指标：

- `duration`: 生成出来的音频时长。
- `RTF`: real-time factor，生成耗时 / 音频时长。`RTF < 1` 才有机会实时播放。

采样率也很关键：16kHz 文件小、速度快；48kHz 更适合高保真输出，但文件和后处理成本更高。

In [ ]:
def pick_torch_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


def download_modelscope_model(model_id):
    from modelscope import snapshot_download

    print(f"Loading ModelScope model: {model_id}")
    model_dir = snapshot_download(model_id)
    print(f"ModelScope cache: {model_dir}")
    return model_dir


def report_wav(path, elapsed_seconds):
    info = sf.info(path)
    duration_seconds = info.frames / info.samplerate if info.samplerate else 0.0
    rtf = elapsed_seconds / duration_seconds if duration_seconds else float("inf")
    print(f"saved: {path}")
    print(
        f"sample_rate={info.samplerate}, duration={duration_seconds:.2f}s, "
        f"elapsed={elapsed_seconds:.2f}s, rtf={rtf:.2f}"
    )
    return {"path": str(path), "sample_rate": info.samplerate, "duration_seconds": duration_seconds, "elapsed_seconds": elapsed_seconds, "rtf": rtf}


def save_wav(path, waveform, sample_rate, elapsed_seconds):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, waveform, sample_rate)
    return report_wav(path, elapsed_seconds)


def save_wav_bytes(path, wav_bytes, elapsed_seconds):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(bytes(wav_bytes))
    return report_wav(path, elapsed_seconds)


def save_modelscope_tts_output(path, output, elapsed_seconds):
    from modelscope.outputs import OutputKeys

    wav = output
    if isinstance(output, dict):
        wav = output.get(OutputKeys.OUTPUT_WAV) or output.get("output_wav") or output.get("wav")

    if wav is None:
        details = output.keys() if isinstance(output, dict) else type(output)
        raise ValueError(f"ModelScope TTS output does not contain WAV data: {details}")

    if isinstance(wav, (bytes, bytearray)):
        return save_wav_bytes(path, wav, elapsed_seconds)

    if isinstance(wav, (str, os.PathLike)):
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(wav, path)
        return report_wav(path, elapsed_seconds)

    sample_rate = int(output.get("sample_rate", 16_000)) if isinstance(output, dict) else 16_000
    return save_wav(path, wav, sample_rate, elapsed_seconds)

## 3. 小模型：ModelScope Sambert + HiFiGAN 的 TTS 链路

这个小模型适合先验证大陆网络环境下的下载和运行链路，模块边界也很容易讲清楚：

```text
text
-> ModelScope text_to_speech pipeline
-> Sambert 声学模型生成声学特征
-> HiFiGAN vocoder 生成 waveform
-> 保存 16kHz WAV
```

这里不再依赖 Hugging Face 的模型仓库或 `datasets` 数据集下载；后续中文、多语言、音色控制和参考音频克隆放到 VoxCPM2 章节看。

In [ ]:
if RUN_SMALL_TTS:
    from modelscope.pipelines import pipeline
    from modelscope.utils.constant import Tasks

    print("device hint:", pick_torch_device())
    small_model_dir = download_modelscope_model(SMALL_TTS_MODEL_ID)
    small_tts = pipeline(task=Tasks.text_to_speech, model=small_model_dir)
    print("text:", SMALL_TEXT)

In [ ]:
if RUN_SMALL_TTS:
    start = time.perf_counter()
    small_output = small_tts(input=SMALL_TEXT)
    elapsed = time.perf_counter() - start

    small_tts_metrics = save_modelscope_tts_output(
        OUTPUT_DIR / "modelscope_sambert_demo.wav",
        small_output,
        elapsed,
    )
    small_tts_metrics

## 4. 从小模型里观察部署问题

小模型已经能暴露 TTS 部署的核心问题：

- 文本前处理会影响读法，比如数字、缩写、标点和多语言文本。
- pipeline 会隐藏一部分内部细节，但仍然能看到声学模型、vocoder、采样率和输出文件这些关键边界。
- vocoder / decoder 是波形质量和速度的重要瓶颈。
- 输出 WAV 之前要明确采样率，否则播放速度、音调和下游处理都会出错。
- RTF 可以把体验和算力联系起来：同一段文本，模型越慢、音频越长，服务并发越难做。

In [ ]:
if RUN_SMALL_TTS:
    print("output type:", type(small_output))
    if isinstance(small_output, dict):
        print("output keys:", list(small_output.keys()))
    print("saved path:", small_tts_metrics["path"])
    print("duration seconds:", round(small_tts_metrics["duration_seconds"], 2))

## 5. 大模型：VoxCPM2 voice design

VoxCPM2 是更大的现代 TTS 模型。官方文档给出的关键信息包括：2B 参数、48kHz 输出、30 种语言、voice design、style control、reference audio cloning，以及 `generate()` API。

在大陆网络环境里，先用 ModelScope 下载模型，再把本地目录传给 `VoxCPM.from_pretrained()`：

```python
from modelscope import snapshot_download
from voxcpm import VoxCPM
model_dir = snapshot_download("OpenBMB/VoxCPM2")
model = VoxCPM.from_pretrained(model_dir, load_denoiser=False)
wav = model.generate(text="...", cfg_value=2.0, inference_timesteps=10)
```

本教程里用 `optimize=False` 保守运行，减少部分 CPU/MPS 环境里 `torch.compile` 的兼容性变量；CUDA 环境可以自行改成默认优化。

In [ ]:
if RUN_VOXCPM2:
    from voxcpm import VoxCPM

    voxcpm2_model_dir = download_modelscope_model(VOXCPM2_MODEL_ID)
    print("Loading VoxCPM2 from local ModelScope cache.")
    voxcpm2 = VoxCPM.from_pretrained(
        voxcpm2_model_dir,
        device="auto",
        load_denoiser=False,
        optimize=False,
    )

    start = time.perf_counter()
    wav = voxcpm2.generate(
        text=VOXCPM_TEXT,
        cfg_value=2.0,
        inference_timesteps=10,
        normalize=True,
    )
    elapsed = time.perf_counter() - start

    voxcpm2_metrics = save_wav(
        OUTPUT_DIR / "voxcpm2_voice_design.wav",
        wav,
        voxcpm2.tts_model.sample_rate,
        elapsed,
    )
    voxcpm2_metrics
else:
    print("Skip VoxCPM2. Change RUN_VOXCPM2 to True when you have time and disk/GPU budget.")

## 6. VoxCPM2 参考音频克隆

如果你有一段干净的参考音频，可以用 `reference_wav_path`。VoxCPM2 的参考音频克隆不要求提供参考音频的逐字 transcript；参考音频主要控制“谁在说”，括号里的文本控制语气、速度和风格。

参考音频建议：

- 5 到 30 秒。
- 尽量干净、少混响、少背景噪声。
- 说话人授权明确，不要克隆没有授权的真实人物声音。

In [ ]:
REFERENCE_WAV_PATH = "speech/reference_speaker.wav"

if RUN_VOXCPM2 and Path(REFERENCE_WAV_PATH).exists():
    start = time.perf_counter()
    cloned = voxcpm2.generate(
        text="(语气更轻松，稍微慢一点) 这是一段参考音频克隆的测试。",
        reference_wav_path=REFERENCE_WAV_PATH,
        cfg_value=2.0,
        inference_timesteps=10,
        normalize=True,
    )
    elapsed = time.perf_counter() - start
    save_wav(
        OUTPUT_DIR / "voxcpm2_reference_clone.wav",
        cloned,
        voxcpm2.tts_model.sample_rate,
        elapsed,
    )
elif RUN_VOXCPM2:
    print(f"No reference file found: {REFERENCE_WAV_PATH}")
else:
    print("Reference cloning section is skipped because RUN_VOXCPM2 is False.")

## 7. 小模型和大模型怎么对比

| 维度 | ModelScope Sambert + HiFiGAN | VoxCPM2 |
| --- | --- | --- |
| 教学价值 | 先跑通大陆可下载的基础 TTS pipeline | 更接近现代 TTS 产品能力 |
| 输出采样率 | 16kHz | 48kHz |
| 多语言 | 默认中文 TTS | 支持多语言语音合成 |
| 音色条件 | 模型内置音色 | voice design、reference audio、prompt audio |
| 部署成本 | 较低 | 高，模型权重、冷启动和显存压力更明显 |
| 适合场景 | 入门、链路学习、轻量实验 | 高质量合成、克隆、风格控制、多语言验证 |

面试或项目里可以这样讲：小模型负责让我在可访问的 ModelScope 链路里理解 TTS 的基础结构，大模型负责验证真实产品里的质量、控制、成本和稳定性问题。

## 8. 部署检查清单

上线 TTS 服务前，至少检查：

- 输入：文本规范化、语言识别、敏感内容、长文本切分。
- 音色：reference audio 授权、质量、时长、去噪策略。
- 推理：设备、dtype、冷启动、RTF、并发、队列超时。
- 输出：采样率、响度、静音裁剪、文件格式、流式播放。
- 稳定性：长文本漂移、生成过短/过长、噪声、重复、异常重试。
- 成本：GPU 显存、模型下载体积、缓存目录、请求峰值和降级模型。

## 参考资料

- ModelScope Sambert + HiFiGAN TTS model: https://modelscope.cn/models/damo/speech_sambert-hifigan_tts_zh-cn_16k
- ModelScope VoxCPM2 model: https://modelscope.cn/models/OpenBMB/VoxCPM2
- VoxCPM2 Quick Start: https://voxcpm.readthedocs.io/en/latest/quickstart.html
- VoxCPM2 Usage Guide: https://voxcpm.readthedocs.io/en/latest/usage_guide.html
- VoxCPM2 model overview: https://voxcpm.readthedocs.io/en/latest/models/voxcpm2.html